## A simple CNN in Pytorch
Following [Michael Li](https://towardsdatascience.com/build-a-fashion-mnist-cnn-pytorch-style-efb297e22582), who explains how to do all of this. See also the Pytorch tutorials, where Li probably got his information.

In [ ]:
# Plotting and numeric libraries
import torch
import bokeh
from bokeh.plotting import figure, output_notebook, show, output_file
from bokeh.models import Label
import numpy as np
import pandas as pd
import tqdm.auto as tqdm
import tarfile

# sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

print(f"""Using Torch version {torch.__version__}.  
        CUDA is {'available' if torch.cuda.is_available() else 'not available'}. 
        MPS is {'available' if torch.backends.mps.is_available() else 'not available'}""")
gpu = 'mps' if torch.backends.mps.is_available() else 'cuda'
cpu = 'cpu'

print(f"Using bokeh version {bokeh.__version__}.")

print(f"Using pandas version {pd.__version__}.")

In [ ]:
output_notebook()
device = gpu

In [ ]:
# Build the neural network, expand on top of nn.Module
class Network(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # define layers
        self.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        ## takes a 28 x 28 matrix and produces 6 x 24 x 24 matrices
        self.conv2 = torch.nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        ## takes 6 24 x 24 matrices and producs 12 x 20 x 20 matrices

        self.fc1 = torch.nn.Linear(in_features=12*4*4, out_features=120)
        self.fc2 = torch.nn.Linear(in_features=120, out_features=60)
        self.out = torch.nn.Linear(in_features=60, out_features=10)

    # define forward function
    def forward(self, t):
        # conv 1
        t = self.conv1(t)
        ## 1 x 28 x 28 goes to 6 x 24 x 24
        t = torch.nn.functional.relu(t)
        t = torch.nn.functional.max_pool2d(t, kernel_size=2, stride=2)
        ## 6 x 24 x 24 goes to 6 x 12 x 12 through pooling

        # conv 2
        t = self.conv2(t)
        ## 6 x 12 x 12 goes to 12 x 8 x 8 
        t = torch.nn.functional.relu(t)
        t = torch.nn.functional.max_pool2d(t, kernel_size=2, stride=2)
        ## pooling 12 x 4 x 4

        # fc1
        t = t.reshape(-1, 12*4*4)
        t = self.fc1(t)
        t = torch.nn.functional.relu(t)
        ## 120 features in to 60 out

        # fc2
        t = self.fc2(t)
        t = torch.nn.functional.relu(t)
        ## 60 in to 10 out

        # output
        t = self.out(t)
        # don't need softmax here since we'll use cross-entropy as activation.

        return t

In [ ]:
with tarfile.open("neural.tgz", "r:gz") as tar:
    # adjust the member name to the CSV inside the archive
    with tar.extractfile("fmnist_data.csv") as f:
        train_images = pd.read_csv(f)
    with tar.extractfile("fmnist_labels.csv") as f:
        train_labels = pd.read_csv(f)
        


train_labels = OneHotEncoder().fit_transform(train_labels).toarray()




In [ ]:
train_images.shape

In [ ]:
Xtrain = torch.tensor(train_images.values, dtype=torch.float32, device=device).reshape((train_images.shape[0], 1, 28, 28))
Ytrain = torch.tensor(train_labels, dtype=torch.float32, device=device).reshape((train_labels.shape[0], train_labels.shape[1]))
criterion = torch.nn.functional.cross_entropy

In [ ]:
def train(model, Xt, Yt):
    """One step through the training loop"""
    # reset the gradient calculations
    
    # reset the gradient calculations
    optimizer.zero_grad()
        
    predicted = model(Xt)
    
    # compute the loss
    loss = criterion(predicted,Yt)
    # compute the gradients by backward propogation
    loss.backward()       
    # adjust the weights
    optimizer.step()   
    

    
    return loss.item()

In [ ]:
def training_loop(model, data, target,threshold=1e-6,max_iter=100000):
    """Run the training loop and return the losses"""
    

    losses = []
    prior_loss=1000000
    for i in tqdm.tqdm(range(max_iter)):
        loss = train(model,data, target)
        losses.append(loss)
        if abs(loss-prior_loss) < threshold:
            break
        prior_loss = loss
        
    return losses
    

In [ ]:
def plot_loss(losses):
    """Plot the losses"""
    f=figure(title="Loss over time",x_axis_label="Epoch",y_axis_label="Loss")
    f.line(x=list(range(len(losses))),y=losses)

    
    return f

In [ ]:
model = Network().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
losses = training_loop(model, Xtrain,Ytrain,threshold=1e-6,max_iter=10000)
show(plot_loss(losses))

In [ ]:
(torch.argmax(torch.nn.functional.softmax(model(Xtrain),dim=1),dim=1)==torch.argmax(Ytrain,dim=1)).sum().item()/Xtrain.shape[0]

In [ ]:
output_file("test.html")
show(plot_loss(losses))

In [ ]:
list(model.named_parameters())